# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [7]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set")

In [9]:
# Constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_GEMINI = 'gemini-2.5-flash-lite'
MODEL_GROQ = 'openai/gpt-oss-120b'

In [6]:
# Initialization

openai = OpenAI()
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"


gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [10]:
system_message = """
You are a technical lead engineer who explains technical concepts clearly with examples.
You will give concise answers which are not too long.
"""

In [20]:
# Map display names to model IDs
MODEL_MAP = {"GPT": MODEL_GPT, "Gemini": MODEL_GEMINI, "Groq": MODEL_GROQ}

model_selector = gr.Dropdown(["GPT", "Gemini", "Groq"], label="Select model", value="GPT")

In [21]:
def chat(message, history, model):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    model_id = MODEL_MAP.get(model, MODEL_GPT)
    print("model: ", model)
    if model == "GPT":
        stream = openai.chat.completions.create(model=model_id, messages=messages, stream=True)
    elif model == "Gemini":
        stream = gemini.chat.completions.create(model=model_id, messages=messages, stream=True)
    elif model == "Groq":
        stream = groq.chat.completions.create(model=model_id, messages=messages, stream=True)
        
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
view = gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[model_selector],
    title="Technical Q&A Assistant",
    description="Ask technical questions about programming. Choose your model above.",
).launch()